In [114]:
import sys
import subprocess
import ctypes
import threading

try:
    import mss
    import numpy as np
    import keyboard
except ModuleNotFoundError:
    print("📦 正在自動安裝 mss, numpy, keyboard...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mss", "numpy", "keyboard"])
    import mss
    import numpy as np
    import keyboard

import pyautogui as pyg
from PIL import ImageGrab
import time

pyg.FAILSAFE = True

# ⚡ Windows 底層直連按鍵
def direct_space():
    """仙人掌短跳"""
    ctypes.windll.user32.keybd_event(0x20, 0, 0, 0)
    time.sleep(0.005)
    ctypes.windll.user32.keybd_event(0x20, 0, 2, 0)

def direct_long_space(hold_time=0.22):
    """翼龍長按大跳"""
    ctypes.windll.user32.keybd_event(0x20, 0, 0, 0)
    time.sleep(hold_time)
    ctypes.windll.user32.keybd_event(0x20, 0, 2, 0)

def direct_down(duration=0.35):
    """翼龍蹲下（保持低姿態直到飛過）"""
    ctypes.windll.user32.keybd_event(0x28, 0, 0, 0)
    time.sleep(duration)
    ctypes.windll.user32.keybd_event(0x28, 0, 2, 0)

print("=== 恐龍自動化 AI（仙人掌專用極窄貼身起跳版）===")
print("請確保 Chrome 視窗最大化，按 Ctrl+0 將縮放定在 100%")
print("--------------------------------------------------")
print("💡 快捷鍵說明（專修仙人掌起跳）：")
print("   [-] 鍵：想更晚跳就按 '-'（每次後移 1.0px）")
print("   [+] 鍵：若撞到想早跳就按 '+'（每次前移 1.0px）")
print("--------------------------------------------------")
print("3 秒後自動啟動...\n")

for i in range(3, 0, -1):
    print(f"{i}...")
    time.sleep(1)

pyg_w, pyg_h = pyg.size()
init_img = ImageGrab.grab()
img_w, img_h = init_img.size
scale = img_w / pyg_w

pyg.click(pyg_w // 2, pyg_h // 2)
direct_space()
time.sleep(0.4)

img = ImageGrab.grab()
dino_x, dino_y = None, None

for y in range(int(img_h * 0.3), int(img_h * 0.75), 3):
    for x in range(int(img_w * 0.05), int(img_w * 0.3), 3):
        p = img.getpixel((x, y))
        if p[0] < 120 and p[1] < 120 and p[2] < 120:
            dino_x, dino_y = x, y
            break
    if dino_x:
        break

if not dino_x:
    print("❌ 找不到小恐龍！")
else:
    # 🎯 專修：預設起算點壓在鼻尖內側 -1.0px，貼身才起跳
    cactus_offset = -1.0   
    flyer_offset = 12.0   # 🦅 翼龍遠距離獨立前瞻 (完全鎖死不改)

    def adjust_cactus_offset():
        global cactus_offset
        while True:
            if keyboard.is_pressed('+') or keyboard.is_pressed('='):
                cactus_offset += 1.0
                print(f"\n📈 仙人掌感應推前: +{cactus_offset:.1f}px (較早跳)")
                time.sleep(0.2)
            elif keyboard.is_pressed('-'):
                cactus_offset -= 1.0
                print(f"\n📉 仙人掌感應縮後: {cactus_offset:.1f}px (較晚跳)")
                time.sleep(0.2)
            time.sleep(0.05)

    t = threading.Thread(target=adjust_cactus_offset, daemon=True)
    t.start()

    print(f"✅ Dino 定位成功:({dino_x}, {dino_y})")

    start_game_time = time.time()

    with mss.mss() as sct:
        try:
            in_air = False
            
            while True:
                elapsed_time = time.time() - start_game_time
                speed_boost = min((elapsed_time / 15.0) * 1.5, 12.0)

                # =========================================================
                # 🌵 獨立監控區域 1：仙人掌專用（極窄 2px 寬度，徹底消除過早進框）
                # =========================================================
                c_left = int(dino_x + (cactus_offset * scale))
                cactus_monitor = {
                    "left": c_left,
                    "top": int(dino_y - (20 * scale)),
                    "width": max(int(2.0 * scale), 1),  # 🎯 寬度大幅縮窄至 2px，強迫仙人掌逼近鼻尖才觸發
                    "height": int(12 * scale)
                }

                # =========================================================
                # 🦅 獨立監控區域 2：翼龍專用（100% 保持鎖定，不碰仙人掌）
                # =========================================================
                f_left = int(dino_x + ((flyer_offset + speed_boost) * scale))
                flyer_monitor = {
                    "left": f_left,
                    "top": int(dino_y - (50 * scale)),
                    "width": max(int(8.0 * scale), 1),
                    "height": int(28 * scale)
                }

                # --- 1. 優先擷取仙人掌獨立區域 ---
                c_img = np.array(sct.grab(cactus_monitor))
                c_gray = c_img[:, :, 0]
                c_bg = c_gray[0, 0]
                c_mask = (c_gray.astype(int) - int(c_bg)) > 20 if c_bg < 120 else (int(c_bg) - c_gray.astype(int)) > 20
                c_pixels = np.sum(c_mask)

                # --- 2. 擷取翼龍獨立區域 ---
                if c_pixels < 2 and not in_air:
                    f_img = np.array(sct.grab(flyer_monitor))
                    f_gray = f_img[:, :, 0]
                    f_bg = f_gray[0, 0]
                    f_mask = (f_gray.astype(int) - int(f_bg)) > 20 if f_bg < 120 else (int(f_bg) - f_gray.astype(int)) > 20
                    f_pixels = np.sum(f_mask)
                else:
                    f_pixels = 0

                # =========================================================
                # 🎯 完全獨立的動作判斷邏輯
                # =========================================================
                if c_pixels < 2 and f_pixels < 2:
                    in_air = False
                elif not in_air:
                    # 🌵 仙人掌獨立邏輯：極致貼身起跳
                    if c_pixels >= 2:
                        direct_space()
                        in_air = True
                        print(f"【🌵 仙人掌貼身短跳】offset: {cactus_offset:.1f}px                     ", end="\r")
                        time.sleep(0.20)

                    # 🦅 翼龍獨立邏輯：維持原樣完全不動
                    elif f_pixels >= 2:
                        y_indices = np.where(f_mask)[0]
                        max_y = y_indices.max()
                        h = f_gray.shape[0]

                        if max_y > int(h * 0.30):
                            direct_long_space(0.22)
                            in_air = True
                            print(f"【🦅 同高翼龍長跳】前瞻: +{flyer_offset + speed_boost:.1f}px           ", end="\r")
                            time.sleep(0.25)
                        else:
                            direct_down(0.35)
                            in_air = True
                            print(f"【🦅 高空翼龍蹲下】                                          ", end="\r")
                            time.sleep(0.10)

        except KeyboardInterrupt:
            print("\n自動化程式已停止。")

=== 恐龍自動化 AI（仙人掌專用極窄貼身起跳版）===
請確保 Chrome 視窗最大化，按 Ctrl+0 將縮放定在 100%
--------------------------------------------------
💡 快捷鍵說明（專修仙人掌起跳）：
   [-] 鍵：想更晚跳就按 '-'（每次後移 1.0px）
   [+] 鍵：若撞到想早跳就按 '+'（每次前移 1.0px）
--------------------------------------------------
3 秒後自動啟動...

3...
2...
1...
✅ Dino 定位成功:(129, 345)


C:\Users\mying\AppData\Local\Temp\ipykernel_632\2767226322.py:103: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


【🌵 仙人掌貼身短跳】offset: -1.0px                           
自動化程式已停止。
